# **TUNING BEST CYCLE 3 CONFIG**

In [13]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **1. Configs**

In [14]:
"""
Configuration module.

get_base_config() defines shared default values.
get_experiment_configs() defines all named experimental configurations.

To add a new experiment, add a new entry to get_experiment_configs()
using **base to inherit all default values and overriding only what changes.
"""

def get_base_config():
    return {
        "model_name":     "Maltehb/danish-bert-botxo",
        "max_length":     512,
        "batch_size":     16,
        "epochs":         20,
        "lr":             2e-5,
        "warmup_ratio":   0.1,
        "weight_decay":   0.0,
        "threshold":      0.5,
        "dropout":        0.0,
        "seed":           42,
        "output_dir":     "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models"
    }


def get_experiment_configs():
    base = get_base_config()

    # ------------------------------------------------------------------
    # Stage 1: Learning rate, batch size, and weight decay
    #
    # ------------------------------------------------------------------

    stage1 = {}

    learning_rates = [2e-5, 3e-5, 5e-5]
    batch_sizes = [8, 16]
    weight_decays = [0.0, 0.01, 0.1]

    for lr in learning_rates:
        for bs in batch_sizes:
            for wd in weight_decays:
                # Format hyperparameters for experiment name
                lr_str = f"lr{int(lr * 1e5)}e5"
                bs_str = f"bs{bs:02d}"
                wd_str = f"wd{int(wd * 100):03d}"

                exp_name = f"s1_{lr_str}_{bs_str}_{wd_str}"

                stage1[exp_name] = {
                    **base,
                    "resampling": "ros",
                    "target_ratio": 1.0,
                    "lr": lr,
                    "batch_size": bs,
                    "weight_decay": wd,
                    "stage": 1
                }


    # ------------------------------------------------------------------
    # Stage 2: Systematic hyperparameter optimization with augmented data
    # Comprehensive grid search testing interactions between LR, WD,
    # dropout, and batch size, with gradient clipping to address instability
    # ------------------------------------------------------------------

    stage2 = {}

    learning_rates = [1.5e-5, 2e-5]
    weight_decays = [0.01, 0.05, 0.1]
    dropouts = [0.1, 0.2, 0.3]
    batch_sizes = [16, 32]

    for lr in learning_rates:
        for wd in weight_decays:
            for do in dropouts:
                for bs in batch_sizes:
                    # Format hyperparameters for experiment name
                    lr_str = f"lr{int(lr * 1e5):02d}e5"
                    wd_str = f"wd{int(wd * 100):03d}"
                    do_str = f"do{int(do * 10):01d}"
                    bs_str = f"bs{bs:02d}"

                    exp_name = f"s2_{lr_str}_{wd_str}_{do_str}_{bs_str}"

                    stage2[exp_name] = {
                        **base,
                        "lr": lr,
                        "batch_size": bs,
                        "weight_decay": wd,
                        "dropout": do,
                        "resampling": None,
                        "target_ratio": 0.0,
                        "max_grad_norm": 1.0,
                        "stage": 2
                    }


    # ------------------------------------------------------------------
    # Stage 3: Resampling with ROS and RUS
    # ------------------------------------------------------------------
    best_lr = 2e-5
    best_bs = 16
    best_wd = 0.10
    best_dropout = 0.2

    stage3 = {
        "s3_ros_050": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": "ros",
            "target_ratio": 0.50,
            "stage": 3
        },

        "s3_ros_075": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": "ros",
            "target_ratio": 0.75,
            "stage": 3
        },

        "s3_rus_050": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": "rus",
            "target_ratio": 0.50,
            "stage": 3
        },

        "s3_rus_075": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": "rus",
            "target_ratio": 0.75,
            "stage": 3
        },

        "s3_rus_100": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": "rus",
            "target_ratio": 1.0,
            "stage": 3
        }
    }

    # ------------------------------------------------------------------
    # Stage 4: Longer training
    # ------------------------------------------------------------------
    best_resampling = "rus"
    best_ratio = 0.75

    stage4 = {
        "s4_50epochs": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": best_resampling,
            "target_ratio": best_ratio,
            "epochs": 50,
            "stage": 4
        },

        "s4_100epochs": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": best_resampling,
            "target_ratio": best_ratio,
            "epochs": 100,
            "stage": 4
        }
    }

    # ------------------------------------------------------------------
    # Stage 5: Weighted Loss
    # ------------------------------------------------------------------
    stage5 = {
        "s5_pos_weight": {
            **base,
            "lr": best_lr,
            "batch_size": best_bs,
            "weight_decay": best_wd,
            "dropout": best_dropout,
            "max_grad_norm": 1.0,
            "resampling": None,
            "target_ratio": 0.0,
            "epochs": 50,
            "pos_weight": True,
            "stage": 5
        }
    }


    return {**stage1, **stage2, **stage3, **stage4, **stage5}


# **2. Data**

In [15]:
import torch
from torch.utils.data import Dataset
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ============================================
# DATALOADER
# ============================================
class ClimateDataset(Dataset):
    """
    PyTorch Dataset for binary climate claim classification.

    Args:
        df (pd.DataFrame): Input dataframe containing text and labels.
        tokenizer:         HuggingFace tokenizer.
        max_len (int):     Maximum sequence length.
    """

    def __init__(self, df, tokenizer, max_len):
        self.texts     = df["ArticleText"].fillna("").tolist()
        self.labels    = df["has_claim"].values.astype(int)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids":      tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }


# ============================================
# RESAMPLER
# ============================================
def apply_resampling(df, target_ratio, method='ros'):
    """
    Apply resampling using imblearn library.

    Args:
        df: DataFrame with 'has_claim' column
        target_ratio: Target minority/majority ratio (0.0 = no resampling)
        method: 'ros' (random oversampling) or 'rus' (random undersampling)

    Returns:
        Resampled DataFrame
    """
    if target_ratio == 0.0:
        print("No resampling applied (target_ratio=0.0)")
        return df.copy()

    # Separate features and labels
    X = df.drop(columns=['has_claim']).reset_index(drop=True)
    y = df['has_claim'].reset_index(drop=True)

    # Select resampler
    if method == 'ros':
        sampler = RandomOverSampler(
            sampling_strategy=target_ratio
        )
    elif method == 'rus':
        sampler = RandomUnderSampler(
            sampling_strategy=target_ratio
        )
    else:
        raise ValueError(f"Unknown resampling method: {method}")

    # Apply resampling
    X_resampled, y_resampled = sampler.fit_resample(X, y)

    # Reconstruct DataFrame
    df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
    df_resampled['has_claim'] = y_resampled

    # Print statistics
    original_counts = y.value_counts()
    resampled_counts = y_resampled.value_counts()
    print(f"\nResampling ({method}, target_ratio={target_ratio}):")
    print(f"  Original:  {original_counts[1]} positive / {original_counts[0]} negative")
    print(f"  Resampled: {resampled_counts[1]} positive / {resampled_counts[0]} negative")
    print(f"  New ratio: {resampled_counts[1] / resampled_counts[0]:.3f}")

    return df_resampled


# ============================================
# WEIGHTED LOSS
# ============================================
def compute_pos_weight(df, label_col='has_claim'):
    """
    Compute positive class weight for imbalanced binary classification.

    Args:
        df: Training DataFrame
        label_col: Name of label column

    Returns:
        torch.Tensor: pos_weight for BCEWithLogitsLoss
    """
    labels = df[label_col].values
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(labels),
        y=labels
    )
    pos_weight = torch.tensor([class_weights[1] / class_weights[0]], dtype=torch.float32)

    print(f"Class distribution: {pd.Series(labels).value_counts().to_dict()}")
    print(f"Computed pos_weight: {pos_weight.item():.3f}")

    return pos_weight


# **3. Model**

In [16]:
"""
Model module for binary classification.

Loads pretrained Danish BERT and configures it for binary classification
with optional dropout adjustment.
"""

from transformers import AutoConfig, AutoModelForSequenceClassification


def create_model(model_name, config):
    """
    Initialize a pretrained transformer model for binary classification.

    Args:
        model_name (str): HuggingFace model identifier
        config (dict): Experiment configuration


    Returns:
        model: Configured HuggingFace model
    """
    # Load base configuration
    model_config = AutoConfig.from_pretrained(model_name)

    model_config.num_labels = 2
    model_config.problem_type = "single_label_classification"

    # Apply dropout if specified
    dropout = config.get("dropout", None)

    if dropout is not None and dropout > 0.0:
        model_config.hidden_dropout_prob = dropout
        model_config.attention_probs_dropout_prob = dropout
        print(f"Model configured with dropout={dropout}")

    # Load pretrained model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=model_config
    )

    return model

# **4. Evaluation**

In [17]:
"""
Evaluation module for binary classification.
Uses fixed threshold=0.5 for all evaluations.
"""

import numpy as np
from scipy.special import softmax
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score
)


def compute_metrics(eval_pred):
    """
    Compute metrics for HuggingFace Trainer.

    Args:
        eval_pred: Tuple of (predictions, labels) from Trainer

    Returns:
        dict: Metrics (f1, precision, recall, balanced_accuracy)
    """
    logits, labels = eval_pred

    # Convert logits to probabilities and apply threshold
    probs = softmax(logits, axis=1)[:, 1]  # probability of positive class
    preds = (probs > 0.5).astype(int)

    return {
        "f1": f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
    }

# **5. Training**

In [18]:
"""
Training module for binary classification.
Creates HuggingFace Trainer with early stopping.
"""

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback


def create_trainer(model, train_dataset, val_dataset, config, compute_metrics):
    """
    Create HuggingFace Trainer for binary classification.

    Args:
        model: HuggingFace model
        train_dataset: Training dataset
        val_dataset: Validation dataset
        config: Experiment configuration dict
        compute_metrics: Metrics function

    Returns:
        Trainer instance
    """
    training_args = TrainingArguments(
        output_dir=config["output_dir"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        learning_rate=config["lr"],
        warmup_ratio=config["warmup_ratio"],
        weight_decay=config["weight_decay"],
        max_grad_norm=config["max_grad_norm"],
        seed=config["seed"],

        # Evaluation and saving
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
    )

    return Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )

In [19]:

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import torch.nn as nn


class WeightedTrainer(Trainer):
    """Custom Trainer with weighted binary cross-entropy loss."""

    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Use BCEWithLogitsLoss with pos_weight
        if self.pos_weight is not None:
            loss_fct = nn.BCEWithLogitsLoss(
                pos_weight=self.pos_weight.to(logits.device)
            )
        else:
            loss_fct = nn.BCEWithLogitsLoss()

        # Reshape for binary classification
        loss = loss_fct(
            logits.view(-1, 2)[:, 1],  # Get positive class logits
            labels.float()
        )

        return (loss, outputs) if return_outputs else loss


def create_trainer(model, train_dataset, val_dataset, config, compute_metrics, pos_weight=None):
    """
    Create HuggingFace Trainer for binary classification.

    Args:
        model: HuggingFace model
        train_dataset: Training dataset
        val_dataset: Validation dataset
        config: Experiment configuration dict
        compute_metrics: Metrics function
        pos_weight: Optional tensor for weighted loss

    Returns:
        Trainer instance
    """
    training_args = TrainingArguments(
        output_dir=config["output_dir"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        learning_rate=config["lr"],
        warmup_ratio=config["warmup_ratio"],
        weight_decay=config["weight_decay"],
        seed=config["seed"],

        # Evaluation and saving
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
    )

    # Use WeightedTrainer if pos_weight provided, else standard Trainer
    if pos_weight is not None:
        return WeightedTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            pos_weight=pos_weight,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
        )
    else:
        return Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
        )

# **6. Plotting**

In [20]:
# define plot to examining training behaviour
def plot_training_curves(log_path, save_path):
    """Generate 2-panel training curve visualization for binary classification"""

    import matplotlib.pyplot as plt
    import pandas as pd

    logs = pd.read_csv(log_path)
    train_logs = logs[logs['loss'].notna()]
    val_logs = logs[logs['eval_loss'].notna()]

    # Set style
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Panel 1: Loss curves
    axes[0].plot(train_logs['epoch'], train_logs['loss'],
                 marker='o', linewidth=2, markersize=5, label='Train', color='#2E86AB')
    axes[0].plot(val_logs['epoch'], val_logs['eval_loss'],
                 marker='s', linewidth=2, markersize=5, label='Validation', color='#A23B72')
    axes[0].set_xlabel('Epoch', fontsize=11)
    axes[0].set_ylabel('Loss', fontsize=11)
    axes[0].legend(frameon=True, fancybox=True, shadow=False)
    axes[0].set_title('Training & Validation Loss', fontsize=12, pad=10)

    # Panel 2: F1 Score
    axes[1].plot(val_logs['epoch'], val_logs['eval_f1'],
                 marker='o', linewidth=2, markersize=5, color='#06A77D')
    axes[1].set_xlabel('Epoch', fontsize=11)
    axes[1].set_ylabel('F1 Score', fontsize=11)
    axes[1].set_title('Validation F1 Score', fontsize=12, pad=10)
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

# **7. Grid Search Runner**

In [21]:
import gc
import json
import os
from datetime import datetime
import torch
import pandas as pd
from transformers import AutoTokenizer
from transformers.trainer_utils import get_last_checkpoint


# File paths
TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/data/processed/cycle3/train_augmented.parquet"
VAL_PATH   = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/data/processed/cycle3/val_manual_200.parquet"
SAVE_PATH  = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/training03/results_binary.json"

# Load data
train_df = pd.read_parquet(TRAIN_PATH)
val_df   = pd.read_parquet(VAL_PATH)

# Set to relevant stage
STAGE = 3

# Load already completed experiments once before the loop
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, "r") as f:
        all_results = json.load(f)
    completed = {r["experiment"] for r in all_results}
else:
    all_results = []
    completed = set()

configs = get_experiment_configs()

for experiment_name, config in configs.items():

    if STAGE is not None and config.get("stage") != STAGE:
        continue

    # Skip experiments already saved to results.json
    if experiment_name in completed:
        print(f"Skipping {experiment_name} — already completed.")
        continue

    print(f"\n{'='*60}")
    print(f"Running experiment: {experiment_name}")
    print(f"{'='*60}")

    config["output_dir"] = os.path.join(
        "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models",
        experiment_name
    )

    tokenizer = AutoTokenizer.from_pretrained(config["model_name"])

    # Apply resampling or compute class weights
    use_class_weight = config.get("use_class_weight", False)
    pos_weight = None

    if use_class_weight:
        # Use class weighting instead of resampling
        train_df_sampled = train_df.copy()
        pos_weight = compute_pos_weight(train_df_sampled)
        print("Using class weighting (no resampling)")

    elif config.get("target_ratio", 0.0) == 0.0:
        train_df_sampled = train_df.copy()

    else:
        train_df_sampled = apply_resampling(
            train_df,
            target_ratio=config["target_ratio"],
            method=config["resampling"]
        )

    # Create datasets
    train_dataset = ClimateDataset(train_df_sampled, tokenizer, config["max_length"])
    val_dataset   = ClimateDataset(val_df, tokenizer, config["max_length"])

    # Create model
    model = create_model(config["model_name"], config)

    # Create trainer (with pos_weight if using class weighting)
    trainer = create_trainer(
        model, train_dataset, val_dataset,
        config, compute_metrics, pos_weight=pos_weight
    )

    os.makedirs(config["output_dir"], exist_ok=True)
    checkpoint = get_last_checkpoint(config["output_dir"])
    if checkpoint is not None:
        print(f"Resuming from checkpoint: {checkpoint}")
        trainer.train(resume_from_checkpoint=checkpoint)
    else:
        trainer.train()

    # Save training logs
    log_path = os.path.join(config["output_dir"], "training_logs.csv")
    pd.DataFrame(trainer.state.log_history).to_csv(log_path, index=False)

    plot_path = os.path.join(config["output_dir"], "training_curves.png")
    plot_training_curves(log_path, plot_path)
    print(f"Saved training curves to {plot_path}")

    # Evaluate on validation set
    val_metrics = trainer.evaluate(val_dataset)

    results = {
        "balanced_accuracy": val_metrics["eval_balanced_accuracy"],
        "f1": val_metrics["eval_f1"],
        "precision": val_metrics["eval_precision"],
        "recall": val_metrics["eval_recall"],
        "loss": val_metrics["eval_loss"],
        "run_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "experiment": experiment_name,
        "config": config
    }

    all_results.append(results)
    with open(SAVE_PATH, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\nExperiment {experiment_name} completed.")
    print(f"  F1: {results['f1']:.4f}")
    print(f"  Precision: {results['precision']:.4f}")
    print(f"  Recall: {results['recall']:.4f}")
    print(f"  Balanced Accuracy: {results['balanced_accuracy']:.4f}")

    # Clear GPU memory between runs
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("All experiments completed.")
print("="*60)

Skipping s3_ros_050 — already completed.
Skipping s3_ros_075 — already completed.
Skipping s3_rus_050 — already completed.
Skipping s3_rus_075 — already completed.

Running experiment: s3_rus_100

Resampling (rus, target_ratio=1.0):
  Original:  786 positive / 1710 negative
  Resampled: 786 positive / 786 negative
  New ratio: 1.000
Model configured with dropout=0.2


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Maltehb/danish-bert-botxo
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Saved training curves to /content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models/s3_rus_100/training_curves.png



Experiment s3_rus_100 completed.
  F1: 0.5000
  Precision: 0.4118
  Recall: 0.6364
  Balanced Accuracy: 0.7920

All experiments completed.


In [22]:
"""
Results analysis.
Generates a summary table and training curve plot for a given stage.
Set STAGE to the stage number you want to inspect.
"""
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

SAVE_PATH      = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/training03/results_binary.json"
BASE_MODEL_DIR = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models"
STAGE          = 4

with open(SAVE_PATH, "r") as f:
    all_results = json.load(f)

# Summary table
rows = []
for r in all_results:
    if r["config"].get("stage") != STAGE:
        continue

    rows.append({
        "experiment":        r["experiment"],
        "lr":                r["config"]["lr"],
        "batch_size":        r["config"]["batch_size"],
        "weight_decay":      r["config"].get("weight_decay", 0.0),
        "dropout":           r["config"].get("dropout", "None"),
        "resampling":        r["config"].get("resampling", "None"),
        "use_class_weight":  r["config"].get("use_class_weight", False),
        "f1":                round(r["f1"], 4),
        "balanced_accuracy": round(r["balanced_accuracy"], 4),
        "precision":         round(r["precision"], 4),
        "recall":            round(r["recall"], 4),
    })

summary_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
print(f"Stage {STAGE} results — sorted by F1:\n")
print(summary_df.to_string(index=False))

Stage 4 results — sorted by F1:

  experiment      lr  batch_size  weight_decay  dropout resampling  use_class_weight     f1  balanced_accuracy  precision  recall
 s4_50epochs 0.00002          16           0.1      0.2        rus             False 0.6667             0.8908     0.5625  0.8182
s4_100epochs 0.00002          16           0.1      0.2        rus             False 0.6087             0.8051     0.5833  0.6364
